<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Data Audit
EDA: column detection, spend share, correlation, VIF, seasonality.

In [ ]:
!pip install git+https://github.com/google/meridian.git altair jsonschema==3.2.0 -q

In [ ]:
!pip install jsonschema==3.2.0
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import IPython
import jsonschema
import importlib

# Reload jsonschema to ensure the newly installed version is active
importlib.reload(jsonschema)

from meridian import constants
from meridian.data import load
from meridian.data import test_utils
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import optimizer
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
from meridian.analysis import formatter

# check if GPU is available
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

## Load Data

In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

gcs_file_path = '/content/drive/MyDrive/MMM2_optimized_synthetic_for_meridian_clean.csv'

# Keep on_bad_lines='skip' in case there are still parsing issues from previous attempts
data = pd.read_csv(gcs_file_path, on_bad_lines='skip')
print(data.columns)

# Identify columns that contain spend-related keywords (e.g., 'spend', 'cost', 'budget')
media = [
    col for col in data.columns
    if any(keyword in col.lower() for keyword in ['spend', 'cost', 'budget'])
]

# Exclude media channels with zero total spend
media = [col for col in media if data[col].sum() > 0]


print("Selected Media Channels:", media)

impressions =  [
    col for col in data.columns
    if "impression" in col.lower() or "impresseion" in col.lower()
]

print("Impressions List:", impressions)


# Convert 'date' column to datetime objects using the specified format yyyy/mm/dd
try:
    data['date'] = pd.to_datetime(data['date'], format='%Y/%m/%d', errors='coerce')
except Exception as e:
    print(f"Could not convert 'date' to datetime: {e}")

# Drop rows where date conversion failed (i.e., 'date' is NaT)
data = data.dropna(subset=['date'])

# Identify the output (revenue) column(s)
output = [
    col for col in data.columns
    if 'revenue' in col.lower()
]

# Create a time index t
data['t'] = range(len(data))

# Separate features (X) and target (y)
X = data.drop(output, axis=1)
y = data[output[0]]  # if there is only one revenue column


display(data.head())

## EDA — Spend Share, Correlation, VIF

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ----------------------------
# 1. Spend Share Analysis
# ----------------------------
print("******** Spend Share Analysis ********")
if media:
    # Calculate total spend across all identified media channels.
    total_media_spend = data[media].sum().sum()
    # Compute spend share for each media channel.
    spend_share = data[media].sum() / total_media_spend
    spend_share_df = spend_share.reset_index()
    spend_share_df.columns = ['Media_Channel', 'Spend_Share']
    print(spend_share_df)

    # Plot a bar chart of spend share.
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Spend_Share', y='Media_Channel',
                data=spend_share_df.sort_values(by='Spend_Share', ascending=False))
    plt.title("Spend Share by Media Channel")
    plt.xlabel("Spend Share")
    plt.ylabel("Media Channel")
    plt.show()
else:
    print("No media channels available for spend share analysis.")


# ----------------------------
# 2. Correlation Analysis between Inputs and Output (ecommerce_revenue)
# ----------------------------
print("\n******** Correlation Analysis ********")
revenue_col = output[0] if output and output[0] in data.columns else 'ecommerce_revenue'
print("Using revenue column:", revenue_col)

numeric_data = data.select_dtypes(include=[np.number])
corr_matrix = numeric_data.corr()

if revenue_col in corr_matrix.columns:
    corr_with_rev = corr_matrix[revenue_col].drop(labels=[revenue_col]).sort_values(ascending=False)
    print("Correlation of inputs with {}:".format(revenue_col))
    print(corr_with_rev)

    # Identify top 10 features most correlated (by absolute value) with the revenue.
    top_features = corr_with_rev.abs().sort_values(ascending=False).head(10).index.tolist() + [revenue_col]

    # Plot a heatmap displaying correlations among these top features.
    plt.figure(figsize=(10, 8))
    sns.heatmap(numeric_data[top_features].corr(), annot=True, cmap='coolwarm', center=0)
    plt.title("Heatmap of Top Correlated Features with {}".format(revenue_col))
    plt.show()
else:
    print("Revenue column not found in the correlation matrix.")


# ----------------------------
# 3. Multicollinearity Analysis (Variance Inflation Factor)
# ----------------------------
print("\n******** Multicollinearity Analysis ********")
# Use the feature set X defined earlier; select only numeric columns.
X_numeric = X.select_dtypes(include=[np.number]).drop(columns=['t'], errors='ignore')
if not X_numeric.empty:
    vif_df = pd.DataFrame()
    vif_df["Feature"] = X_numeric.columns
    # Calculate VIF for each feature.
    vif_df["VIF"] = [variance_inflation_factor(X_numeric.values, i)
                     for i in range(X_numeric.shape[1])]
    print(vif_df.sort_values(by="VIF", ascending=False))
else:
    print("No numeric features available in X for multicollinearity analysis.")


# ----------------------------
# 4. Check for Variables with Less Than 15 Records
# ----------------------------
print("\n******** Variables with Less Than 15 Records ********")
low_record_columns = {col: data[col].count() for col in data.columns if data[col].count() < 15}
if low_record_columns:
    for col, cnt in low_record_columns.items():
        print(f"Column '{col}' has only {cnt} recorded non-null entries.")
else:
    print("All variables have at least 15 records.")


## Channel Mappings

In [ ]:
def create_channel_mappings(media, impressions):
    """
    Generate mapping dictionaries for media spend and impressions columns.

    Parameters:
        media (list of str): List of column names with spend/cost data.
                             Expected to have columns ending with '_spend'. # Reverted docstring to original expectation
        impressions (list of str): List of column names with impressions data.
                                   Expected to have columns ending with 'impression' or 'impresseion'. # Updated docstring
                                   Note: If this list contains spend columns (as in the original call),
                                   the resulting 'impressions_mapping' might not be meaningful for impression data.

    Returns:
        tuple: Two dictionaries:
            - correct_media_spend_to_channel: Mapping of spend/cost column names to simplified channel names.
            - correct_media_to_channel: Mapping of impressions column names to simplified channel names.
    """
    correct_media_spend_to_channel = {}
    for col in media:
        if col.endswith("spend"):
            channel_name = col[:-len("spend")]
        else:
            channel_name = col
        correct_media_spend_to_channel[col] = channel_name

    correct_media_to_channel = {}
    for col in impressions:
        col_lower = col.lower()
        if col_lower.endswith("impression"):
            channel_name = col[:-len("impression")]
        elif col_lower.endswith("impresseion"):
            channel_name = col[:-len("impresseion")]
        else:
            channel_name = col
        correct_media_to_channel[col] = channel_name

    return correct_media_spend_to_channel, correct_media_to_channel

# Correctly call create_channel_mappings passing media and impressions lists
cost_mapping, impressions_mapping = create_channel_mappings(media, impressions)

print("Cost Mapping:", cost_mapping)
print("Impressions Mapping:", impressions_mapping)

## Seasonality Features

In [ ]:
skip_seasonality = False

if not skip_seasonality:
  def create_seasonality_features(data: pd.DataFrame, date_column: str, output_variable: str, yearly_seasonality: int) -> pd.DataFrame:
    """
    Creates seasonality effect features for modeling based on the yearly seasonality using the statsmodels library.
    Handles ValueError by using additive seasonality.

    Parameters:
    - data (pd.DataFrame): The input dataframe.
    - date_column (str): The name of the date column.
    - output_variable (str): The name of the output variable.
    - yearly_seasonality (int): The yearly seasonality period (e.g., 52 for weekly, 365 for daily).

    Returns:
    - pd.DataFrame: The dataframe with added seasonality features.
    """

    # Ensure the date column is in datetime format
    data[date_column] = pd.to_datetime(data[date_column])

    # Sort the data by date
    data = data.sort_values(by=date_column)

    # Set the date column as index for seasonal decomposition
    data.set_index(date_column, inplace=True)

    # try multiplicative if not then additive
    try:
        decomposition = seasonal_decompose(data[output_variable], model='multiplicative', period=yearly_seasonality)
    except ValueError as e:
        print(f"Multiplicative decomposition failed: {e}. Switching to additive model.")
        decomposition = seasonal_decompose(data[output_variable], model='additive', period=yearly_seasonality)


    # Add seasonal component as a feature
    data[f'seasonal_{output_variable}'] = decomposition.seasonal

    # Reset index to retain the date column
    data.reset_index(inplace=True)

    return data

# Example usage
# Use the actual output column name identified from the data (stored in output[0])
  data = create_seasonality_features(
    data,
    date_column='date',
    output_variable=output[0],  # Modified to use the actual revenue column name
    yearly_seasonality=52
  )

  data.head()

else:
    print("Skipping seasonality feature creation due to insufficient data.")